In [2]:
import sys

In [3]:
project_root = "/home/krosen5/Court Case Sequence Transformer Model/court_case_transformer_project"

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(sys.path[:3])

['/home/krosen5/Court Case Sequence Transformer Model/court_case_transformer_project', '/home/krosen5/anaconda3/envs/legalsequencetransformer/lib/python313.zip', '/home/krosen5/anaconda3/envs/legalsequencetransformer/lib/python3.13']


In [4]:
import pickle
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, normalize
from scipy.spatial.distance import cdist
import umap
import ot  # pip install POT
 
from utils.cache_directory_utils import CacheDirectoryManager
from data.load_data import mount_drive, load_data
from models.transformer_model import CaseTransformerModel
from models.transformer_creation import create_vocabularies_from_data
from config import get_model_config
from analysis.pca_vectors import (
    extract_all_pre_classifier_vectors,
    reduce_vectors_with_pca,
    run_kmeans,
    plot_pca_by_clusters,
    plot_pca_label_grid,
    plot_multilabel_cluster_heatmap,
    _plot_pca_by_fold,
)
from notebook_helpers import prepare_sequences_and_labels
from models.interpretability import CaseTransformerInterpretability
import torch
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
# -- 1. Load everything ONCE in the main process --------------
from data.load_data import mount_drive, load_data
from models.transformer_creation import create_vocabularies_from_data
from config import get_model_config
import torch
import pickle
import h5py
import types
import threading
from pathlib import Path
from models.transformer_model import CaseTransformerModel
from models.interpretability import CaseTransformerInterpretability
from utils.cache_directory_utils import CacheDirectoryManager
from config import get_model_config
import pickle
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, normalize
import umap

from utils.cache_directory_utils import CacheDirectoryManager
from data.load_data import mount_drive, load_data
from models.transformer_model import CaseTransformerModel
from models.transformer_creation import create_vocabularies_from_data
from config import get_model_config
from analysis.pca_vectors import (
    extract_all_pre_classifier_vectors,
    reduce_vectors_with_pca,
    run_kmeans,
    plot_pca_by_clusters,
    plot_pca_label_grid,
    plot_multilabel_cluster_heatmap,
    _plot_pca_by_fold,
)
from models.magnitude_interpretability_chunk import _run_magnitude_chunk
import multiprocessing
multiprocessing.set_start_method('spawn', force=True)
from models.interpretability_helpers import (
    build_age_quartile_bins,
    build_prior_case_quartile_bins,
)

In [5]:
import matplotlib.pyplot as plt
from scipy.linalg import svd
def align_embeddings(embeddings, labels, fold_indices):
    """
    Aligns 5 separate fold coordinate systems using label centroids
    and then performs a unified PCA.
    
    Args:
        embeddings: (N, D) array of embeddings (e.g., 2500 x 768)
        labels: (N, 20) binary matrix of multi-label outcomes
        fold_indices: (N,) array indicating which fold (0-4) each case belongs to
    """
    n_folds = 5
    n_labels = labels.shape[1]
    dim = embeddings.shape[1]
    
    # 1. Calculate Centroids for each fold (1-5)
    fold_centroids = {}
    for f in range(1, n_folds + 1):
        f_mask = (fold_indices == f)
        f_embeddings = embeddings[f_mask]
        f_labels = labels[f_mask]
        
        centroids = []
        for l in range(n_labels):
            label_mask = (f_labels[:, l] == 1)
            if np.any(label_mask):
                centroids.append(f_embeddings[label_mask].mean(axis=0))
            else:
                # If a label is missing in a fold, use a zero vector for now
                centroids.append(np.zeros(dim))
        fold_centroids[f] = np.array(centroids)

    # 2. Align Folds 2-5 to Fold 1 (the anchor space)
    target_centroids = fold_centroids[1]
    aligned_embeddings = np.copy(embeddings)

    for f in range(2, n_folds + 1):
        source_centroids = fold_centroids[f]
        
        # Filter out any labels that are missing (zero) in either target or source
        # to ensure the SVD isn't skewed by empty rows
        valid_mask = (np.sum(np.abs(target_centroids), axis=1) > 0) & \
                     (np.sum(np.abs(source_centroids), axis=1) > 0)
        
        if not np.any(valid_mask):
            print(f"Warning: No shared labels found for Fold {f}. Skipping rotation.")
            continue

        # SVD Procrustes on valid landmarks only
        H = target_centroids[valid_mask].T @ source_centroids[valid_mask]
        U, S, Vt = svd(H)
        R = U @ Vt
        
        # Apply the rotation to every embedding in this fold
        f_mask = (fold_indices == f)
        aligned_embeddings[f_mask] = embeddings[f_mask] @ R.T
        
    return aligned_embeddings

In [6]:
# ===========================================================
# pca_from_saved_models.py
# Loads fold models + data from disk, runs PCA with
# per-fold normalisation - no retraining needed
# Config: balanced_config_8.json
# ===========================================================



# -- 1. Config (from balanced_config_8.json) ------------------
BASE_CACHE   = Path('/data/krosen5/cache/transformers/balanced_8')
MODELS_DIR   = BASE_CACHE / 'saved_models'
OUTPUT_DIR   = BASE_CACHE / 'pca_normalised'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_FOLDS      = 5
RANDOM_SEED  = 42
N_CLUSTERS   = 6
bert_out_dim = 768
cuda_index   = 0

parameters_dict = {
    'model_validation_strategy': 'k_fold_cross_validation',
    'k_fold_num_folds': 5,
    'k_fold_shuffle': True,
    'random_seed': 42,
    'skip_duplicate_labels': False,
    'skip_duplicate_labels_on_import': False,
    'bert_model_to_use': 'bert-base-uncased',
    'bert_batch_size': 32,
    'sampling_strategy': 'balanced',
    'weighted_loss_function': False,
    'use_mixed_precision': True,
    'use_learning_rate_scheduler': True,
    'gradient_clip_norm': 1.0,
    'number_of_epochs_for_training': 20,
    'batch_size_for_training': 16,
    'low_frequency_threshold': 400,
    'folder_name_directory': 'balanced_8',
}

# -- 2. Device ------------------------------------------------
computation_device = torch.device(
    f'cuda:{cuda_index}' if torch.cuda.is_available() else 'cpu'
)
print(f'Using device: {computation_device}')

# -- 3. Cache manager -----------------------------------------
model_cache_manager = CacheDirectoryManager(base_path=BASE_CACHE)

# -- 4. Load label processor ----------------------------------
label_processor_path = MODELS_DIR / 'label_processor.pkl'
with open(label_processor_path, 'rb') as f:
    label_processor = pickle.load(f)
print(f'Loaded label processor ({len(label_processor.label_names)} labels)')

# -- 5. Load sequences ----------------------------------------
sequences_path = model_cache_manager.get_cache_dir('datasets') / 'sequences_and_labels.pkl'
with open(sequences_path, 'rb') as f:
    sequences, label_sequences_raw = pickle.load(f)
print(f'Loaded {len(sequences)} sequences')

# -- 6. Rebuild the same KFold splits (same seed = same splits)
filtered_training_data = list(zip(sequences, label_sequences_raw))

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
fold_val_sets = []
for train_idx, val_idx in kf.split(filtered_training_data):
    fold_val_sets.append([filtered_training_data[i] for i in val_idx])
print(f'Recreated {N_FOLDS} fold splits')

# -- 7. Load court_data (needed to build vocabularies) --------
drive_root_folder = mount_drive('tulane-courts-shared/v2')
court_data = load_data(drive_root_folder)

# -- 8. Derive correct temporal_dim from checkpoint -----------
model_config = get_model_config()
bond_dim     = model_config.NUM_BOND_CLASSES   # 9

sample_pt    = MODELS_DIR / 'k_fold_fold_1' / 'final_model_mlb_fold_1.pt'
sample_sd    = torch.load(sample_pt, map_location='cpu')
fuse_in      = sample_sd['fuse.0.weight'].shape[1]  # 1548

FIXED_DIMS       = 32 + 192 + 768 + 32 + 96 + 192 + 1 + 1  # 1314
correct_temporal_dim = fuse_in - FIXED_DIMS - (bond_dim * 2)  # 216
print(f'Using temporal_dim={correct_temporal_dim}  '
      f'(fuse_in={fuse_in}, bond_dim={bond_dim})')

# Build vocabs once - same for all folds
charge_vocab, mag_vocab = create_vocabularies_from_data(court_data)

Using device: cuda:0
Loaded label processor (20 labels)
Loaded 2533 sequences
Recreated 5 fold splits
[OK] Found Google Drive mount at /data/krosen5/gdrive-all
[OK] Google Drive mount is accessible
[OK] Data directory found: /data/krosen5/gdrive-all/tulane-courts-shared/v2
Using temporal_dim=216  (fuse_in=1548, bond_dim=9)
Creating charge vocabulary from charges DataFrame with 2139 unique charges
Found 5 unique magistrate sections in mag data.
Creating magistrate vocabulary of size 5


In [7]:
all_val_sequences = []
all_val_labels    = []
for fold_idx in range(N_FOLDS):
    fold_seqs, fold_labels = prepare_sequences_and_labels(fold_val_sets[fold_idx])
    all_val_sequences.extend(fold_seqs)
    all_val_labels.extend(fold_labels)

In [8]:
BEST_FOLD_IDX = 0

In [9]:
n_gpus = torch.cuda.device_count()
print(f'Found {n_gpus} GPUs, running with parallel={n_gpus > 1}')

Found 4 GPUs, running with parallel=True


In [10]:
drive_root   = mount_drive('tulane-courts-shared/v2')
court_data   = load_data(drive_root)
charge_vocab, mag_vocab = create_vocabularies_from_data(court_data)
del court_data  # free memory - workers don't need it

# Save vocabs to temp file so workers can load without rebuilding
vocab_path = BASE_CACHE / 'temp_vocabs.pkl'
with open(vocab_path, 'wb') as f:
    pickle.dump((charge_vocab, mag_vocab), f)
print(f'Vocabs saved to {vocab_path}')

# Compute temporal_dim once
model_cfg    = get_model_config()
bond_dim     = model_cfg.NUM_BOND_CLASSES
pt_path      = BASE_CACHE / 'saved_models' / f'k_fold_fold_{BEST_FOLD_IDX + 1}' / \
               f'final_model_mlb_fold_{BEST_FOLD_IDX + 1}.pt'
sample_sd    = torch.load(pt_path, map_location='cpu')
fuse_in      = sample_sd['fuse.0.weight'].shape[1]
FIXED_DIMS   = 32 + 192 + 768 + 32 + 96 + 192 + 1 + 1
temporal_dim = fuse_in - FIXED_DIMS - (bond_dim * 2)
print(f'temporal_dim={temporal_dim}')

# Pool all fold val sets
all_val_sequences, all_val_labels = [], []
for fold_idx in range(N_FOLDS):
    fold_seqs, fold_labels = prepare_sequences_and_labels(fold_val_sets[fold_idx])
    all_val_sequences.extend(fold_seqs)
    all_val_labels.extend(fold_labels)
print(f'Pooled {len(all_val_sequences)} val sequences')


# -- Split and dispatch ----------------------------------------
magnitude_dir = OUTPUT_DIR / 'magnitude_study'
magnitude_dir.mkdir(parents=True, exist_ok=True)

age_bins        = build_age_quartile_bins(all_val_sequences)
prior_case_bins = build_prior_case_quartile_bins(all_val_sequences)


chunks    = np.array_split(range(len(all_val_sequences)), n_gpus)
args_list = [
    (
        [all_val_sequences[i] for i in chunk],
        [all_val_labels[i]    for i in chunk],
        gpu_idx,
        BEST_FOLD_IDX,
        str(BASE_CACHE),
        str(magnitude_dir),
        gpu_idx,
        str(vocab_path),
        temporal_dim,
        bond_dim,
        age_bins,
        prior_case_bins,
    )
    for gpu_idx, chunk in enumerate(chunks)
]

ctx = multiprocessing.get_context('spawn')
with ProcessPoolExecutor(max_workers=n_gpus, mp_context=ctx) as pool:
    list(pool.map(_run_magnitude_chunk, args_list))


# -- Merge results ---------------------------------------------
from scipy import stats

features   = ["race", "gender", "age", "original_bond", "prior_case_count"]
merged_agg = {}
merged_raw = {}

for feat in features:
    dfs = [
        pd.read_pickle(magnitude_dir / f'chunk_{i}_{feat}.pkl')
        for i in range(n_gpus)
        if (magnitude_dir / f'chunk_{i}_{feat}.pkl').exists()
    ]
    if not dfs:
        print(f'  No chunks found for {feat}')
        continue

    combined          = pd.concat(dfs, ignore_index=True)
    merged_raw[feat]  = combined

    agg = (combined
           .groupby(['original_value', 'swapped_value', 'label'])
           .agg(
               mean_abs_delta=('abs_delta',  'mean'),
               std_abs_delta =('abs_delta',  'std'),
               mean_delta    =('delta_prob', 'mean'),
               std_delta     =('delta_prob', 'std'),
               n             =('abs_delta',  'count'),
           )
           .reset_index())

    agg['se_abs_delta']   = agg['std_abs_delta'] / np.sqrt(agg['n'])
    agg['se_delta']       = agg['std_delta']     / np.sqrt(agg['n'])
    agg['ci95_half_width_abs_delta']    = agg.apply(
        lambda r: stats.t.ppf(0.975, df=max(r['n'] - 1, 1)) * r['se_abs_delta']
                  if r['n'] > 1 else np.nan, axis=1)
    agg['ci95_half_width_delta']  = agg.apply(
        lambda r: stats.t.ppf(0.975, df=max(r['n'] - 1, 1)) * r['se_delta']
                  if r['n'] > 1 else np.nan, axis=1)
    agg['ci95_abs_delta'] = agg.apply(
        lambda r: f"{r['mean_abs_delta']:.4f} +/- {r['ci95_half_width_abs_delta']:.4f}"
                  if r['n'] > 1 else f"{r['mean_abs_delta']:.4f} (n=1)", axis=1)
    agg['ci95_delta']     = agg.apply(
        lambda r: f"{r['mean_delta']:.4f} +/- {r['ci95_half_width_delta']:.4f}"
                  if r['n'] > 1 else f"{r['mean_delta']:.4f} (n=1)", axis=1)

    merged_agg[feat] = agg
    agg.to_excel(magnitude_dir / f'counterfactual_{feat}.xlsx', index=False)

# Clean up temp vocab file
vocab_path.unlink(missing_ok=True)

# Summary
summary_rows = [
    {'feature': feat, 'mean_abs_delta': df['mean_abs_delta'].mean()}
    for feat, df in merged_agg.items()
]
summary_df = pd.DataFrame(summary_rows).sort_values('mean_abs_delta', ascending=False)
print('\nFeature Sensitivity Summary:')
print(summary_df.to_string(index=False))
summary_df.to_excel(magnitude_dir / 'counterfactual_summary.xlsx', index=False)
print('\nAll chunks merged')

[OK] Found Google Drive mount at /data/krosen5/gdrive-all
[OK] Google Drive mount is accessible
[OK] Data directory found: /data/krosen5/gdrive-all/tulane-courts-shared/v2
Creating charge vocabulary from charges DataFrame with 2139 unique charges
Found 5 unique magistrate sections in mag data.
Creating magistrate vocabulary of size 5
Vocabs saved to /data/krosen5/cache/transformers/balanced_8/temp_vocabs.pkl
temporal_dim=216
Pooled 2533 val sequences
  Age quartile bins: ['Q1 (<23)', 'Q2 (23-30)', 'Q3 (30-37)', 'Q4 (37+)']
  Prior case count distribution: 49.0% zero cases
  Prior case count bins: ['none (0)', 'low (1-2)', 'medium (3-3)', 'high (4+)']
  [GPU 0] Vocabs loaded
  [GPU 1] Vocabs loaded
  [GPU 2] Vocabs loaded
  [GPU 3] Vocabs loaded
  [GPU 0] Model loaded
Date range updated: 2008 to 2025

COUNTERFACTUAL MAGNITUDE STUDY
  Samples : 634
  Features: race, gender, age, original_bond, prior_case_count

  [GPU 1] Model loaded
Date range updated: 2008 to 2025

COUNTERFACTUAL MAGNI

Counterfactual study (parallel) [GPU 1]:   0%|         | 0/633 [00:00<?, ?it/s]

Counterfactual study (parallel) [GPU 2]:   0%|         | 0/633 [00:00<?, ?it/s]


Counterfactual study (parallel) [GPU 3]:   0%|         | 0/633 [00:00<?, ?it/s]


Counterfactual study (parallel) [GPU 3]:   0%| | 1/633 [00:00<06:12,  1.69it/s]


Counterfactual study (parallel) [GPU 3]:   0%| | 3/633 [00:00<02:12,  4.75it/s]

Counterfactual study (parallel) [GPU 2]:   0%| | 1/633 [00:00<08:19,  1.26it/s]


Counterfactual study (parallel) [GPU 0]:   0%| | 1/634 [00:01<11:52,  1.13s/it]


Counterfactual study (parallel) [GPU 3]:   1%| | 7/633 [00:01<01:23,  7.47it/s]


Counterfactual study (parallel) [GPU 3]:   1%| | 9/633 [00:01<01:13,  8.47it/s]


Counterfactual study (parallel) [GPU 0]:   0%| | 2/634 [00:01<07:19,  1.44it/s]


Counterfactual study (parallel) [GPU 0]:   0%| | 3/634 [00:01<04:43,  2.22it/s]


Counterfactual study (parallel) [GPU 3]:   3%| | 18/633 [00:01<00:42, 14.41it/s


Counterfactual stu

Tables saved to /data/krosen5/cache/transformers/balanced_8/pca_normalised/magnitude_study/chunk_2





Counterfactual study (parallel) [GPU 0]:  86%|#| 548/634 [03:46<02:13,  1.55s/i

Figure(1000x400)


...

Figure(350x1473)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)


Counterfactual study (parallel) [GPU 0]:  93%|#| 591/634 [04:26<01:07,  1.58s/i

Figure(350x697)
Figure(800x400)

Feature Sensitivity Summary:
         feature  mean_abs_delta
prior_case_count        0.002777
   original_bond        0.002691
          gender        0.001052
             age        0.001045
            race        0.000639


Counterfactual study (parallel) [GPU 0]: 100%|#| 634/634 [05:00<00:00,  2.11it/


Tables saved to /data/krosen5/cache/transformers/balanced_8/pca_normalised/magnitude_study/chunk_0
Figure(1000x400)
Figure(1000x400)
Figure(1000x400)
Figure(1000x400)
Figure(1000x400)
Figure(3200x400)
Figure(3200x400)
Figure(3200x400)
Figure(3200x400)
Figure(3200x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(350x1105)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(350x553)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(350x921)
Figure(630x490)
Figure(630x490)
Figure(630x490)
Figure(630x490)
Figure(630x490)
Figure(630x490)
Figure(630x490)
Figure(630x490)
Figure(350x1473)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(600x400)
Figure(350x697)
Figure(80

In [ ]:
models = []

for fold_idx in range(N_FOLDS):
    print(f'\n{"-"*60}')
    print(f'  Fold {fold_idx + 1} / {N_FOLDS}')
    print(f'{"-"*60}')

    pt_path = MODELS_DIR / f'k_fold_fold_{fold_idx + 1}' / \
              f'final_model_mlb_fold_{fold_idx + 1}.pt'

    model = CaseTransformerModel(
        charge_vocab=charge_vocab,
        mag_vocab=mag_vocab,
        bert_cache_dir=model_cache_manager.get_cache_dir('bert'),
        bert_out_dim=bert_out_dim,
        num_labels=model_config.num_labels,
        bond_dim=bond_dim,
        temporal_dim=correct_temporal_dim,
        cache_manager=model_cache_manager,
    )

    model.load_state_dict(torch.load(pt_path, map_location=computation_device))
    model.eval()
    model.to(computation_device)

    models.append(model)

    print(f'  Loaded {pt_path.name}')


------------------------------------------------------------
  Fold 1 / 5
------------------------------------------------------------
  Loaded final_model_mlb_fold_1.pt

------------------------------------------------------------
  Fold 2 / 5
------------------------------------------------------------
  Loaded final_model_mlb_fold_2.pt

------------------------------------------------------------
  Fold 3 / 5
------------------------------------------------------------
  Loaded final_model_mlb_fold_3.pt

------------------------------------------------------------
  Fold 4 / 5
------------------------------------------------------------
  Loaded final_model_mlb_fold_4.pt

------------------------------------------------------------
  Fold 5 / 5
------------------------------------------------------------
  Loaded final_model_mlb_fold_5.pt


In [12]:
prepare_sequences_and_labels(fold_val_sets[0])


([[{'proceeding_id': 'mag_197177',
    'type': 'mag',
    'date': Timestamp('2017-12-28 00:00:00'),
    'text': '           FIRST APPEARANCE HEARING IN MAGISTRATE COURT SECTION M5          \n           BOND SET FOR $3,000.00                                           \n           SCHEDULED -PROGRESS REPORT ON 1/25/2018 AT  3:00 PM, SECT M5     \n           SCHEDULED -RULE TO SHOW CAUSE ON 2/26/2018 AT  3:00 PM, SECT M2  \n           THE DEFENDANT WAS SERVED FOR THE SECOND APPEARANCE DATE          \n           THE COURT APPOINTED THE OPD TO REPRESENT THE DEFENDANT IN THIS   \n           MATTER THE APPOINTMENT WILL LAST AS LONG AS THE DEFENDANT IS     \n           INCARCERATED IF THE DEFENDANT IS RELEASED THEY MUST OBTAIN       \n           PRIVATE COUNSEL OR APPLY FOR REPRESENTATION AT THE OPD OFFICE    \n           THE COURT ISSUED A NON DOMESTIC STAY AWAY ORDER IN THIS MATTER   \n           ROR THIBODEAUX                                                   ',
    'docket_id': '570388',
 

In [13]:
model1 = models[0]


In [14]:
import threading
import h5py

def _make_thread_safe_text_emb(text_emb):
    """Patch a StreamingCachedBertEmbedding instance to use thread-local HDF5 handles."""
    text_emb._local = threading.local()

    def _ensure_h5_open(self):
        if not hasattr(self._local, 'h5') or self._local.h5 is None:
            self._local.h5           = h5py.File(self.embedding_file, 'r')
            self._local.embedding_ds = self._local.h5['embeddings']

    def forward(self, proceeding_id):
        import torch
        device = next(self.parameters()).device if list(self.parameters()) else torch.device('cpu')
        if proceeding_id and (proceeding_id.startswith('initial_') or proceeding_id == 'INITIAL_STATE'):
            return torch.zeros(1, self.out_dim, device=device)
        embedding_idx = self.index.get(proceeding_id)
        if embedding_idx is None:
            return torch.zeros(1, self.out_dim, device=device)
        self._ensure_h5_open()
        embedding = self._local.embedding_ds[embedding_idx]
        return torch.as_tensor(embedding, device=device).unsqueeze(0)

    import types
    text_emb._ensure_h5_open = types.MethodType(_ensure_h5_open, text_emb)
    text_emb.forward         = types.MethodType(forward, text_emb)
    return text_emb

# Apply before running the study
models[0].text_emb = _make_thread_safe_text_emb(models[0].text_emb)
print('Patched text_emb to use thread-local HDF5 handles')

Patched text_emb to use thread-local HDF5 handles


In [ ]:
# -- 9. Run counterfactual magnitude study -------------------


# Pool all fold val sets



print(f'Pooled {len(all_val_sequences)} val sequences across {N_FOLDS} folds')

BEST_FOLD_IDX = 0  # change to best fold

# Save + force config to match checkpoint
_saved_min_year = model_config.min_year
_saved_max_year = model_config.max_year
print(f'  Saving config state: min_year={_saved_min_year}, max_year={_saved_max_year}, '
      f'temporal_dim={model_config.temporal_dim}')
model_config.update_date_range(2008, 2025)
print(f'  Forced config: min_year={model_config.min_year}, max_year={model_config.max_year}, '
      f'temporal_dim={model_config.temporal_dim}')

try:
    interpreter = CaseTransformerInterpretability(
        model=models[BEST_FOLD_IDX],
        label_processor=label_processor,
        verbose=False,
        device=computation_device,
    )
    print(f'Created interpreter for fold {BEST_FOLD_IDX + 1}')
    magnitude_results = interpreter.run_counterfactual_magnitude_study(
        val_sequences=all_val_sequences,
        val_labels=all_val_labels,
        output_dir=OUTPUT_DIR / 'magnitude_study',
        max_samples=len(all_val_sequences),
        show_plots=False,
        parallel=True,
    )

    print('Magnitude study complete')
    print(magnitude_results['aggregated']['race'])

finally:
    # Restore config whether study succeeded or failed
    model_config.update_date_range(_saved_min_year, _saved_max_year)
    print(f'\nConfig restored: min_year={model_config.min_year}, '
          f'max_year={model_config.max_year}, temporal_dim={model_config.temporal_dim}')

Pooled 2533 val sequences across 5 folds
  Saving config state: min_year=1990, max_year=2025, temporal_dim=432
Date range updated: 2008 to 2025
  Forced config: min_year=2008, max_year=2025, temporal_dim=216
Created interpreter for fold 1

COUNTERFACTUAL MAGNITUDE STUDY
  Samples : 2533
  Features: race, gender, age, original_bond, prior_case_count

  Age quartile bins: ['Q1 (<23)', 'Q2 (23-30)', 'Q3 (30-37)', 'Q4 (37+)']
  Prior case count distribution: 49.0% zero cases
  Prior case count bins: ['none (0)', 'low (1-2)', 'medium (3-3)', 'high (4+)']


Counterfactual study (parallel) [GPU 0]:   0%|          | 7/2533 [00:03<24:01,  1.75it/s]


In [ ]:
if sequences and label_sequences_raw:
    if parameters_dict['model_validation_strategy'] == 'train_val_split':
        interpreter = interpretability_results['interpreter']
        
        magnitude_results = interpreter.run_counterfactual_magnitude_study(
            val_sequences=val_sequences,
            val_labels=val_labels_raw,
            output_dir=model_cache_manager.get_cache_dir('interpretability_results') / 'magnitude_study',
            max_samples=len(val_sequences),
        )

    elif parameters_dict['model_validation_strategy'] == 'k_fold_cross_validation':
        # Pool all fold val sets - each case appears in exactly one fold's val set
        all_val_sequences = []
        all_val_labels    = []

        for fold_idx in range(cv_training_results['n_splits']):
            fold_val_data = cv_training_results['fold_results'][fold_idx]['val_data']
            fold_seqs, fold_labels = prepare_sequences_and_labels(fold_val_data)
            all_val_sequences.extend(fold_seqs)
            all_val_labels.extend(fold_labels)

        # Use the best fold's interpreter
        best_fold_idx   = cv_training_results['best_fold_idx']
        best_interpreter = intepretability_results_per_fold[best_fold_idx]['interpreter']

        magnitude_results = best_interpreter.run_counterfactual_magnitude_study(
            val_sequences=all_val_sequences,
            val_labels=all_val_labels,
            output_dir=model_cache_manager.get_cache_dir('interpretability_results') / 'magnitude_study',
            max_samples=len(all_val_sequences),
        )

In [ ]:
# -- 9. Extract embeddings from each fold ---------------------

# Save + force config to match checkpoint
_saved_min_year = model_config.min_year
_saved_max_year = model_config.max_year
print(f'  Saving config state: min_year={_saved_min_year}, max_year={_saved_max_year}, '
      f'temporal_dim={model_config.temporal_dim}')
model_config.update_date_range(2008, 2025)
print(f'  Forced config: min_year={model_config.min_year}, max_year={model_config.max_year}, '
      f'temporal_dim={model_config.temporal_dim}')

all_vectors      = []
all_final_labels = []
all_case_labels  = []
all_metadata     = []

for fold_idx in range(N_FOLDS):
    print(f'\n{"-"*60}')
    print(f'  Fold {fold_idx + 1} / {N_FOLDS}')
    print(f'{"-"*60}')

    pt_path = MODELS_DIR / f'k_fold_fold_{fold_idx + 1}' / \
              f'final_model_mlb_fold_{fold_idx + 1}.pt'

    model = CaseTransformerModel(
        charge_vocab=charge_vocab,
        mag_vocab=mag_vocab,
        bert_cache_dir=model_cache_manager.get_cache_dir('bert'),
        bert_out_dim=bert_out_dim,
        num_labels=model_config.num_labels,
        bond_dim=bond_dim,
        temporal_dim=correct_temporal_dim,
        cache_manager=model_cache_manager,
    )
    model.load_state_dict(torch.load(pt_path, map_location=computation_device))
    model.eval()
    model.to(computation_device)
    print(f'  Loaded {pt_path.name}')

    vectors, final_labels, case_labels_matrix, metadata = \
        extract_all_pre_classifier_vectors(
            model, fold_val_sets[fold_idx], label_processor, computation_device,
            use_teacher_forcing=True, verbose=True,
        )

    for m in metadata:
        m['fold_idx'] = fold_idx + 1

    # Per-fold normalisation
    vectors = StandardScaler().fit_transform(vectors)
    vectors = normalize(vectors, norm='l2')

    all_vectors.append(vectors)
    all_final_labels.extend(final_labels)
    all_case_labels.append(case_labels_matrix)
    all_metadata.extend(metadata)

    del model
    torch.cuda.empty_cache()

# Restore config
model_config.update_date_range(_saved_min_year, _saved_max_year)
print(f'\nConfig restored: min_year={model_config.min_year}, '
      f'max_year={model_config.max_year}, temporal_dim={model_config.temporal_dim}')


# ==============================================================
# -- 10. GW Barycenter + MDS -----------------------------------
# ==============================================================
# The correct way to visualise embeddings from independently
# trained models with no shared samples:
#   1. Compute pairwise distance matrices for each fold
#   2. Find a GW barycenter - a consensus distance matrix
#      that no single fold dominates
#   3. Embed the barycenter into 2D via SMACOF MDS
#   4. Map each fold's samples into those 2D coords via their
#      GW transport plan to the barycenter
# ==============================================================

from sklearn import manifold

def smacof_mds(C, dim=2, max_iter=3000):
    """Embed a distance matrix into 2D using SMACOF MDS."""
    rng = np.random.RandomState(seed=RANDOM_SEED)
    mds = manifold.MDS(
        dim, max_iter=max_iter, eps=1e-9,
        dissimilarity='precomputed', n_init=1
    )
    pos = mds.fit(C).embedding_
    nmds = manifold.MDS(
        2, max_iter=max_iter, eps=1e-9,
        dissimilarity='precomputed',
        random_state=rng, n_init=1,
    )
    return nmds.fit_transform(C, init=pos)


print(f'\n{"="*60}')
print('  GW Barycenter + MDS')
print(f'{"="*60}')

# Pairwise cosine distance matrices for each fold
Cs = [cdist(v, v, metric='cosine').astype(np.float64) for v in all_vectors]
Cs = [c / c.max() for c in Cs]
ps = [ot.unif(len(v)) for v in all_vectors]

# Barycenter size = average fold size
N_bary  = int(np.mean([len(v) for v in all_vectors]))
p_bary  = ot.unif(N_bary)
weights = np.ones(N_FOLDS) / N_FOLDS  # equal weight per fold

print(f'  Computing GW barycenter '
      f'({N_FOLDS} folds -> {N_bary} barycenter nodes)...')
C_bary = ot.gromov.gromov_barycenters(
    N_bary, Cs, ps, p_bary, weights,
    loss_fun='square_loss',
    max_iter=50, tol=1e-4,
    verbose=True,
)
print('Barycenter computed')

# Embed barycenter distance matrix into 2D via SMACOF MDS
print('  Embedding barycenter via MDS...')
bary_2d = smacof_mds(C_bary, dim=2)
print(f'  Barycenter 2D shape: {bary_2d.shape}')

# Map each fold's samples into the barycenter 2D space
# via their GW transport plan - soft weighted average of
# barycenter coordinates
print('  Mapping folds into barycenter space...')
all_2d = []
for fold_idx in range(N_FOLDS):
    print(f'    Fold {fold_idx + 1}...')
    T, log = ot.gromov.gromov_wasserstein(
        Cs[fold_idx], C_bary, ps[fold_idx], p_bary,
        loss_fun='square_loss', log=True, verbose=False,
    )
    print(f'      GW dist to barycenter: {log["gw_dist"]:.4f}')

    row_sums = T.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1e-10, row_sums)
    coords_2d = (T @ bary_2d) / row_sums
    all_2d.append(coords_2d)

projected_2d         = np.vstack(all_2d)
all_final_labels_arr = np.array(all_final_labels)
all_case_labels_arr  = np.vstack(all_case_labels)
fold_ids             = np.array([m['fold_idx'] for m in all_metadata])

print(f'\nCombined (GW barycenter): '
      f'{projected_2d.shape[0]} samples x 2 dims')

In [ ]:
from sklearn.cluster import DBSCAN


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# -- 11. Plots ------------------------------------------------

# Helper: generic 2D scatter with custom axis labels
def scatter_2d(xy, colours, labels_legend, title, output_path,
               xlabel='GW-Barycenter Dim 1', ylabel='GW-Barycenter Dim 2',
               s=18, alpha=0.6):
    fig, ax = plt.subplots(figsize=(12, 9))
    unique = sorted(set(colours))
    cmap = cm.get_cmap('tab10', len(unique))
    for i, val in enumerate(unique):
        mask = np.array(colours) == val
        ax.scatter(xy[mask, 0], xy[mask, 1],
                   c=[cmap(i)], label=str(labels_legend[i]) if labels_legend else str(val),
                   s=s, alpha=alpha, edgecolors='w', linewidths=0.2)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(loc='best', fontsize=9)
    fig.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches='tight')
    print(f'Saved: {output_path}')
    plt.show(); plt.close(fig)


# Fold sanity check
scatter_2d(
    projected_2d, fold_ids.tolist(),
    labels_legend=[f'Fold {i}' for i in range(1, N_FOLDS+1)],
    title='GW Barycenter Embeddings - Coloured by Fold',
    output_path=OUTPUT_DIR / 'gw_bary_by_fold.png',
)

# KMeans clusters
db = DBSCAN(eps=0.08, min_samples=10)
km_labels = db.fit_predict(projected_2d)

n_clusters_found = len(set(km_labels)) - (1 if -1 in km_labels else 0)
n_noise = (km_labels == -1).sum()
print(f'DBSCAN found {n_clusters_found} clusters, {n_noise} noise points')

unique_labels = sorted(set(km_labels))
legend_labels = ['Noise' if l == -1 else f'Cluster {l}' for l in unique_labels]

scatter_2d(
    projected_2d, km_labels.tolist(),
    labels_legend=legend_labels,
    title='GW Barycenter Embeddings - DBSCAN Clusters',
    output_path=OUTPUT_DIR / 'gw_bary_dbscan_cluster.png',
)

# Label grid - one subplot per label
fig, axes = plt.subplots(4, 5, figsize=(25, 20))
axes = axes.flatten()
for i, label_name in enumerate(label_processor.label_names):
    ax = axes[i]
    mask_pos = all_case_labels_arr[:, i] == 1
    mask_neg = ~mask_pos
    n_pos = mask_pos.sum()
    pct = 100 * n_pos / len(mask_pos)
    ax.scatter(projected_2d[mask_neg, 0], projected_2d[mask_neg, 1],
               c='lightgrey', s=8, alpha=0.3)
    ax.scatter(projected_2d[mask_pos, 0], projected_2d[mask_pos, 1],
               s=14, alpha=0.7)
    ax.set_title(f'{label_name[:30]}\n(n={n_pos:,}, {pct:.1f}%)', fontsize=7)
    ax.set_xticks([]); ax.set_yticks([])

# hide unused axes
for j in range(len(label_processor.label_names), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('GW Barycenter K-Fold Embeddings - Label Presence',
             fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'gw_bary_label_grid.png', dpi=200, bbox_inches='tight')
print(f'Saved: {OUTPUT_DIR}/gw_bary_label_grid.png')
plt.show(); plt.close(fig)

# Cluster heatmap
plot_multilabel_cluster_heatmap(
    km_labels, all_case_labels_arr, label_processor.label_names,
    output_path=OUTPUT_DIR / 'gw_bary_multilabel_heatmap.png',
)

print(f'\nDone. Plots saved to {OUTPUT_DIR}')

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

def plot_label_grid_colored_by_cluster(
    projected_2d,
    label_matrix,
    label_names,
    cluster_labels,
    output_path,
    title='GW Barycenter K-Fold Embeddings - Label Presence (colored by cluster)',
    max_cols=5,
):
    label_matrix    = np.asarray(label_matrix)
    cluster_labels  = np.asarray(cluster_labels)

    n_samples, n_labels = label_matrix.shape
    unique_clusters     = sorted(np.unique(cluster_labels))
    n_clusters          = len(unique_clusters)

    cluster_to_color_idx = {cl: i for i, cl in enumerate(unique_clusters)}
    # exclude noise (-1) from colourmap count so colours aren't wasted on it
    n_real_clusters = sum(1 for cl in unique_clusters if cl != -1)
    cmap = cm.get_cmap("tab20" if n_real_clusters <= 20 else "nipy_spectral",
                       max(n_real_clusters, 1))

    n_cols = min(max_cols, n_labels)
    n_rows = math.ceil(n_labels / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = np.atleast_1d(axes).flatten()

    for i, label_name in enumerate(label_names):
        ax = axes[i]

        mask_pos = label_matrix[:, i] == 1
        mask_neg = ~mask_pos

        n_pos = int(mask_pos.sum())
        pct   = 100 * n_pos / n_samples if n_samples > 0 else 0.0

        # Background: samples without this label
        ax.scatter(
            projected_2d[mask_neg, 0],
            projected_2d[mask_neg, 1],
            c='lightgrey', s=8, alpha=0.25,
            edgecolors='none', rasterized=True,
        )

        # Foreground: samples with this label, coloured by cluster
        for cl in unique_clusters:
            mask_cl = cluster_labels == cl
            mask    = mask_pos & mask_cl
            if mask.sum() == 0:
                continue

            # noise points (-1) rendered in dark grey, more transparent
            if cl == -1:
                face_color = 'dimgrey'
                alpha      = 0.35
            else:
                face_color = [cmap(cluster_to_color_idx[cl])]
                alpha      = 0.75

            ax.scatter(
                projected_2d[mask, 0],
                projected_2d[mask, 1],
                c=face_color, s=14, alpha=alpha,
                edgecolors='white', linewidths=0.2,
                rasterized=True,
            )

        ax.set_title(f'{label_name[:30]}\n(n={n_pos:,}, {pct:.1f}%)', fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])

    # Hide unused axes
    for j in range(n_labels, len(axes)):
        axes[j].set_visible(False)

    # Legend - noise gets a fixed dark grey swatch
    handles = [
        plt.Line2D(
            [0], [0],
            marker='o', color='w',
            markerfacecolor='dimgrey' if cl == -1
                            else cmap(cluster_to_color_idx[cl]),
            markeredgecolor='white', markeredgewidth=0.4,
            markersize=7,
            label='Noise' if cl == -1 else f'Cluster {cl}',
        )
        for cl in unique_clusters
    ]

    fig.legend(
        handles=handles,
        loc='upper center',
        bbox_to_anchor=(0.5, 1.01),
        ncol=min(n_clusters, 6),
        fontsize=9,
        frameon=True,
    )

    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.05)
    fig.tight_layout()

    fig.savefig(output_path, dpi=200, bbox_inches='tight')
    print(f'Saved: {output_path}')
    plt.show()
    plt.close(fig)
    
plot_label_grid_colored_by_cluster(
    projected_2d=projected_2d,
    label_matrix=all_case_labels_arr,
    label_names=label_processor.label_names,
    cluster_labels=km_labels,
    output_path=OUTPUT_DIR / 'gw_bary_label_grid_colored_by_cluster.png',
)

In [ ]:
# -- 11. Plots ------------------------------------------------

_plot_pca_by_fold(
    projected_2d, fold_ids, None, N_FOLDS,
    output_path=OUTPUT_DIR / 'gw_bary_by_fold.png'
)

plot_pca_label_grid(
    projected_2d, all_case_labels_arr, label_processor.label_names, None,
    output_path=OUTPUT_DIR / 'gw_bary_label_grid.png',
    title='GW Barycenter K-Fold Embeddings - Label Presence',
)

km_labels, km_model = run_kmeans(
    projected_2d, nclusters=N_CLUSTERS, random_state=RANDOM_SEED
)

plot_pca_by_clusters(
    projected_2d, km_labels, None,
    centroids_2d=km_model.cluster_centers_,
    output_path=OUTPUT_DIR / 'gw_bary_kmeans_cluster.png',
    title='GW Barycenter K-Fold Embeddings - KMeans Clusters',
)

plot_multilabel_cluster_heatmap(
    km_labels, all_case_labels_arr, label_processor.label_names,
    output_path=OUTPUT_DIR / 'gw_bary_multilabel_heatmap.png',
)

print(f'\nDone. Plots saved to {OUTPUT_DIR}')

In [ ]:
from sklearn.neighbors import NearestNeighbors
nbrs = NearestNeighbors(n_neighbors=10).fit(projected_2d)
distances, _ = nbrs.kneighbors(projected_2d)
distances = np.sort(distances[:, -1])
plt.plot(distances)
plt.ylabel('10th nearest neighbour distance')
plt.xlabel('Points sorted by distance')
plt.title('K-distance graph - pick eps at the elbow')
plt.show()